# Graph-Theoretic Synchronization — Interactive Notebook

**Design positive directed-edge weights without solving LMIs or linear-inequality feasibility problems.**

1. **Run the setup cells** (including the editable *Custom dynamics* cell).
2. **Choose an oscillator and graph** in the dashboard.
3. Click **Validate & simulate**. A graph without a spanning out-arborescence is rejected **before** weights are assigned.

The mathematical guarantee requires (i) a spanning out-arborescence and (ii) a **separately established** QUAD inequality for the chosen dynamics and coupling matrix. **This notebook does not prove the QUAD condition for user-defined dynamics.**

Based on [Arokiadoss & Arunkumar, *Physica A* (2026)](https://doi.org/10.1016/j.physa.2026.131508).

In [ ]:
# In Colab, uncomment the next line if packages are missing:
# %pip install -q numpy scipy networkx matplotlib ipywidgets

import csv
import io
import re
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import ipywidgets as widgets
from IPython.display import display, clear_output

# Enable widget rendering on Google Colab, if applicable.
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass

## 1 · Custom dynamical system
Edit the next cell **only** when using the **User-defined** option. Otherwise, leave the example unchanged.

In [ ]:
# ✏️ EDIT THIS CELL only if you select "User-defined" in the dashboard.
# X has shape (state_dimension, number_of_vertices). Return the same shape.
# This two-dimensional contracting oscillator is a working example.

CUSTOM_NAME = "My dynamical system"
CUSTOM_INITIAL_STATE = np.array([1.0, -0.5], dtype=float)

def custom_dynamics(t, X):
    x, y = X
    return np.vstack((-x + y, -x - y))

# For this example, P = diag(1,1), a = 1 satisfies the QUAD inequality
# with c = 2. For any replacement dynamics, prove a valid c > 0 yourself.
# Enter P's diagonal and a in the dashboard below.

## 2 · Mathematical implementation
The three cells below contain the oscillator models, graph-theoretic algorithm, and simulation helpers. They are collapsed to keep the notebook short. Expand them to inspect the complete, commented source.

In [ ]:
# PRESET OSCILLATORS (reference formulas; no editing needed)

def lorenz(t, X):
    x, y, z = X
    return np.vstack((10*(y-x), x*(28-z)-y, x*y-(8/3)*z))

def chua(t, X):
    x, y, z = X
    h = -0.714*x + 0.5*(-1.143+0.714)*(np.abs(x+1)-np.abs(x-1))
    return np.vstack((15.6*(y-x-h), x-y+z, -28*y))

def rossler_like(t, X):
    x, y, z = X
    psi = np.where(x < 2.56, 0.0, 21.43*(x-2.56))
    return np.vstack((-0.03*(0.075*x + 1.5*y + 0.75*z),
                      0.03*(x+0.2*y), 0.03*1.5*(psi-z)))

@dataclass
class Oscillator:
    name: str
    f: object
    x0: np.ndarray
    p: np.ndarray            # Diagonal of P
    a: float
    note: str

def oscillator_choice(selection, p_text, custom_a):
    if selection == 'Lorenz':
        beta, rho, sigma = 8/3, 28.0, 10.0
        a = beta*(beta+1)*(rho+sigma)**2/(16*(beta-1)) - sigma
        return Oscillator('Lorenz', lorenz, np.array([1.,1.,1.]),
                          np.array([1.,0.,0.]), a, 'Published analytical a; x-coupling')
    if selection == 'Chua':
        return Oscillator('Chua', chua, np.array([0.1,0.2,0.3]),
                          np.ones(3), 10., 'Published QUAD: c = 0.6218')
    if selection == 'Rössler-like':
        return Oscillator('Rössler-like', rossler_like, np.array([-1.,-1.,-1.]),
                          np.ones(3), 1., 'Piecewise-linear variant, NOT standard Rössler; c = 0.5074')
    # The selection is user-defined. The user supplies P and a, not c.
    p_values = np.asarray([float(v.strip()) for v in p_text.split(',')], dtype=float)
    cfg = Oscillator(CUSTOM_NAME, custom_dynamics,
                     np.asarray(CUSTOM_INITIAL_STATE, dtype=float),
                     p_values, float(custom_a), 'Custom: QUAD inequality must be proved by user')
    return cfg

def validate_oscillator(cfg):
    if cfg.x0.ndim != 1 or cfg.x0.size < 1 or not np.all(np.isfinite(cfg.x0)):
        raise ValueError('CUSTOM_INITIAL_STATE must be a finite, one-dimensional array.')
    if cfg.p.shape != cfg.x0.shape or not np.all(np.isfinite(cfg.p)):
        raise ValueError(f'P must contain exactly {cfg.x0.size} finite diagonal entries.')
    if not np.all(np.isclose(cfg.p, 0) | np.isclose(cfg.p, 1)) or not np.any(np.isclose(cfg.p, 1)):
        raise ValueError('P must be diagonal with entries 0 or 1, with at least one 1.')
    if not np.isfinite(cfg.a) or cfg.a <= 0:
        raise ValueError('a must be a positive finite number.')
    # Reject incompatible user functions early, before simulation.
    trial = np.repeat(cfg.x0[:, None], 2, axis=1)
    result = np.asarray(cfg.f(0., trial), dtype=float)
    if result.shape != trial.shape or not np.all(np.isfinite(result)):
        raise ValueError('The dynamics function must return a finite array of shape (dimension, vertices).')

In [ ]:
# GRAPH INPUT + GRAPH-THEORETIC WEIGHT ALLOCATION

def _positive_partition(n, k, rng):
    if k == 1:
        return [n]
    cuts = np.sort(rng.choice(np.arange(1, n), size=k-1, replace=False))
    return np.diff(np.r_[0, cuts, n]).tolist()

def random_eligible_graph(n, seed):
    """Construct a random digraph with exactly one source SCC."""
    if n < 2:
        raise ValueError('Use at least 2 vertices.')
    rng = np.random.default_rng(seed)
    k = int(rng.integers(2, min(6, n)+1))
    source_size = int(rng.integers(1, min(5, n-k+1)+1))
    sizes = [source_size] + _positive_partition(n-source_size, k-1, rng)
    labels = list(map(int, rng.permutation(n)))
    blocks, start = [], 0
    G = nx.DiGraph()
    G.add_nodes_from(range(n))
    for size in sizes:
        block = labels[start:start+size]
        blocks.append(block)
        start += size
        if size > 1:
            order = list(rng.permutation(block))
            G.add_edges_from((int(order[j]), int(order[(j+1) % size])) for j in range(size))
            for u in block:
                for v in block:
                    if u != v and not G.has_edge(u, v) and rng.random() < 0.12:
                        G.add_edge(u, v)
    # Rooted condensation graph. Edges go only from earlier to later SCCs.
    for j in range(1, k):
        parent = int(rng.integers(0, j))
        G.add_edge(int(rng.choice(blocks[parent])), int(rng.choice(blocks[j])))
    for i in range(k):
        for j in range(i+1, k):
            if rng.random() < 0.28:
                G.add_edge(int(rng.choice(blocks[i])), int(rng.choice(blocks[j])))
    return G

def parse_edge_text(text, n):
    """Accept `tail,head` or `tail head`, one directed edge per line."""
    G = nx.DiGraph()
    G.add_nodes_from(range(n))
    for lineno, raw in enumerate(text.splitlines(), 1):
        line = raw.split('#', 1)[0].strip()
        if not line:
            continue
        parts = [p for p in re.split(r'[,;\s]+', line) if p]
        if len(parts) == 2 and parts[0].lower() in ('source','tail','from'):
            continue
        if len(parts) != 2:
            raise ValueError(f'Line {lineno}: expected two vertex IDs: tail,head.')
        try:
            u, v = map(int, parts)
        except ValueError:
            raise ValueError(f'Line {lineno}: vertex IDs must be integers.') from None
        if not (0 <= u < n and 0 <= v < n):
            raise ValueError(f'Line {lineno}: vertex IDs must lie between 0 and {n-1}.')
        if u == v:
            raise ValueError(f'Line {lineno}: self-loops are not supported.')
        G.add_edge(u, v)
    if G.number_of_edges() == 0:
        raise ValueError('Please provide at least one directed edge.')
    return G

def graph_from_uploaded_csv(upload, n):
    """Read ipywidgets FileUpload content on versions 7 or 8."""
    value = upload.value
    if not value:
        raise ValueError('Upload a CSV file with tail,head columns first.')
    entry = next(iter(value.values())) if isinstance(value, dict) else value[0]
    content = entry['content']
    data = bytes(content).decode('utf-8-sig')
    return parse_edge_text(data, n)

def source_scc_order(G):
    """Raise if no out-arborescence exists; return source-to-sink SCC order."""
    if G.number_of_nodes() < 2:
        raise ValueError('At least 2 vertices are required.')
    sccs = list(nx.strongly_connected_components(G))
    condensation = nx.condensation(G, sccs)
    sources = [v for v in condensation if condensation.in_degree(v) == 0]
    if len(sources) != 1:
        raise ValueError(f'No spanning out-arborescence: found {len(sources)} source SCCs (need exactly one).')
    order = list(nx.topological_sort(condensation))
    return [set(condensation.nodes[i]['members']) for i in order]

def choose_root(G, nodes, source, rng):
    if source:
        return int(rng.choice(sorted(nodes)))
    incoming = {v: 0 for v in nodes}
    for u, v in G.edges():
        if v in nodes and u not in nodes:
            incoming[v] += 1
    choices = [v for v in nodes if incoming[v] == max(incoming.values()) and incoming[v] > 0]
    return int(rng.choice(sorted(choices)))

def negative_imbalance_weights(G, nodes, root, a):
    """Each root-to-vertex path contributes 2a × (length, ..., 1)."""
    H = G.subgraph(nodes)
    weights = {e: 0. for e in H.edges()}
    path_lengths = []
    for v in sorted(nodes):
        if v == root:
            continue
        path = nx.shortest_path(H, root, v)
        ell = len(path)-1
        path_lengths.append(ell)
        for j, e in enumerate(zip(path[:-1], path[1:])):
            weights[e] += 2*a*(ell-j)
    return weights, path_lengths

def positive_circulation(G, nodes):
    """Cover each internal edge by a directed cycle; the sum has zero imbalance."""
    H = G.subgraph(nodes)
    weights = {e: 0. for e in H.edges()}
    if len(nodes) == 1:
        return weights
    return_paths = {}
    for u, v in H.edges():
        if v not in return_paths:
            return_paths[v] = nx.single_source_shortest_path(H, v)
        path = return_paths[v][u]
        weights[(u, v)] += 1.
        for e in zip(path[:-1], path[1:]):
            weights[e] += 1.
    return weights

def internal_imbalance(nodes, weights):
    d = {v: 0. for v in nodes}  # outgoing minus incoming
    for (u, v), w in weights.items():
        d[u] += w
        d[v] -= w
    return d

def allocate_weights(G, a, seed=0):
    """Assign all weights using SCC paths, circulations, and inter-SCC rules."""
    order = source_scc_order(G)  # validate BEFORE changing G
    G = G.copy()
    for u, v in G.edges():
        G[u][v]['weight'] = 0.
    rng = np.random.default_rng(seed+100003)
    diagnostics = []
    for idx, nodes in enumerate(order):
        root = choose_root(G, nodes, idx == 0, rng)
        weights, lengths = negative_imbalance_weights(G, nodes, root, a)
        cycle = positive_circulation(G, nodes)
        S = sum(lengths)
        # The source SCC needs the additional scaling from the published method.
        factor = (2*a/len(nodes))*(1+S)*S if idx == 0 and len(nodes) > 1 else 1.
        internal = {}
        for edge in G.subgraph(nodes).edges():
            w = weights[edge] + factor*cycle[edge]
            G[edge[0]][edge[1]]['weight'] = float(w)
            internal[edge] = float(w)
        d = internal_imbalance(nodes, internal)
        diagnostics.append(dict(nodes=nodes, root=root, imbalance=d))
    # Allocate incoming edges of each non-source SCC (published Algorithm 2).
    for row in diagnostics[1:]:
        nodes, root = row['nodes'], row['root']
        incoming_by_head = {}
        for u, v in G.edges():
            if u not in nodes and v in nodes:
                incoming_by_head.setdefault(v, []).append((u, v))
        for head, edges in incoming_by_head.items():
            total = a + 0.5*row['imbalance'][root] if head == root else a
            for u, v in edges:
                G[u][v]['weight'] = float(total/len(edges))
    return G, diagnostics

def verify_weights(G, diagnostics, a):
    weights = np.array([G[u][v]['weight'] for u, v in G.edges()], dtype=float)
    positive = bool(weights.size and np.all(np.isfinite(weights)) and np.all(weights > 0))
    imbalances = [row['imbalance'][v] for row in diagnostics for v in row['nodes'] if v != row['root']]
    ok = all(value <= -2*a + 1e-8*max(1, a) for value in imbalances)
    return dict(positive=positive, imbalance=ok, min_weight=float(weights.min()),
                max_weight=float(weights.max()), scc_count=len(diagnostics))

In [ ]:
# SIMULATION + PLOTTING + EXPORT

def adjacency_matrix(G):
    n = len(G)
    A = np.zeros((n, n))
    for u, v, info in G.edges(data=True):
        A[v, u] = info['weight']  # row = receiving vertex
    return A

def simulate(G, cfg, t_end, seed, spread, count=180):
    n, d = G.number_of_nodes(), cfg.x0.size
    A = adjacency_matrix(G)
    degrees = A.sum(axis=1)
    rng = np.random.default_rng(seed+200003)
    X0 = cfg.x0[:, None] + spread*rng.normal(size=(d, n))
    def rhs(t, flat):
        X = flat.reshape((d, n), order='F')
        diff = X@A.T - X*degrees[None, :]
        return (cfg.f(t, X) + cfg.p[:, None]*diff).reshape(-1, order='F')
    t = np.linspace(0, t_end, count)
    sol = solve_ivp(rhs, (0, t_end), X0.reshape(-1, order='F'),
                    t_eval=t, method='LSODA', rtol=1e-6, atol=1e-8)
    if not sol.success or not np.all(np.isfinite(sol.y)):
        raise RuntimeError('Numerical integration failed: ' + sol.message)
    return t, sol.y.T, A

def tree_distances(G, root, series):
    tree = nx.bfs_tree(G, root)
    if tree.number_of_nodes() != len(G):
        raise RuntimeError('Internal error: chosen root cannot reach all vertices.')
    edges = list(tree.edges())
    n = len(G)
    dimension = series.shape[1] // n
    D = np.empty((series.shape[0], len(edges)))
    for i, flat in enumerate(series):
        X = flat.reshape((dimension, n), order='F')
        for j, (u, v) in enumerate(edges):
            D[i,j] = np.linalg.norm(X[:, u]-X[:, v])
    return edges, D

def make_plot(G, t, D, seed):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), layout='constrained')
    # Ignore edge weights in the layout: they can differ by orders of magnitude.
    pos = (nx.kamada_kawai_layout(G, weight=None) if len(G) <= 40
           else nx.spring_layout(G, seed=seed, weight=None))
    nx.draw_networkx(G, pos, ax=axes[0], arrows=True,
                     node_size=160 if len(G)<35 else 65,
                     font_size=7, with_labels=len(G)<=35,
                     width=0.7, arrowsize=10)
    axes[0].set_title(f'Directed network ({len(G)} vertices, {G.number_of_edges()} edges)')
    axes[0].axis('off')
    if D.shape[1] <= 10:
        axes[1].plot(t, D, linewidth=0.8, alpha=0.35)
    axes[1].plot(t, D.mean(axis=1), linewidth=2, label='Mean tree-edge distance')
    axes[1].plot(t, D.max(axis=1), linewidth=1.5, linestyle='--', label='Max tree-edge distance')
    axes[1].set(xlabel='Time', ylabel='State distance', title='Synchronization error')
    axes[1].grid(alpha=0.2)
    axes[1].legend(fontsize=8)
    return fig

def export_results(results, directory='output'):
    folder = Path(directory)
    folder.mkdir(parents=True, exist_ok=True)
    G, t, D, A, edges = [results[k] for k in ('G','t','D','A','edges')]
    with (folder/'directed_edges.csv').open('w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['tail','head','weight'])
        writer.writerows((u,v,G[u][v]['weight']) for u,v in G.edges())
    np.savetxt(folder/'weighted_adjacency.csv', A, delimiter=',')
    with (folder/'tree_edge_distances.csv').open('w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['time'] + [f'{u}->{v}' for u,v in edges])
        writer.writerows([float(ti)] + row.tolist() for ti,row in zip(t,D))
    results['fig'].savefig(folder/'synchronization.png', dpi=150, bbox_inches='tight')
    return folder

## 3 · Interactive dashboard
Run the cell below. **All graph modes validate the out-arborescence condition before weights or simulations are computed.**

In [ ]:
# 🎛️ DASHBOARD — select the oscillator and input graph, then click Run.

oscillator = widgets.Dropdown(options=['Lorenz','Chua','Rössler-like','User-defined'],
                              value='Lorenz', description='Dynamics:', style={'description_width':'initial'})
p_entry = widgets.Text(value='1,1', description='P diagonal:',
                       placeholder='Example: 1,0,1', style={'description_width':'initial'})
a_entry = widgets.FloatText(value=1.0, description='a:', style={'description_width':'initial'})
custom_panel = widgets.VBox([
    widgets.HTML('<small>Define <code>custom_dynamics</code> in the editable cell above. Enter the diagonal of P and a.<br>'
                 'A mathematical proof of the QUAD condition (with some c &gt; 0) is required.</small>'),
    widgets.HBox([p_entry, a_entry]),
])

graph_mode = widgets.Dropdown(options=['Random eligible graph','Paste directed edges','Upload edge-list CSV'],
                              description='Graph:', style={'description_width':'initial'})
n_vertices = widgets.BoundedIntText(value=12, min=2, max=200,
                                    description='Vertices:', style={'description_width':'initial'})
edge_entry = widgets.Textarea(value='0,1\n1,2\n2,0\n2,3\n3,4\n4,3\n4,5\n5,6\n6,7\n7,8\n8,9\n9,10\n10,11',
                              placeholder='One tail,head pair per line',
                              description='Edges:', style={'description_width':'initial'},
                              layout=widgets.Layout(width='460px', height='110px'))
upload = widgets.FileUpload(accept='.csv', multiple=False, description='Upload CSV')
edge_help = widgets.HTML('<small>Vertices are labelled 0 to N−1. Each row is <code>tail,head</code>; isolated vertices are included via N.</small>')
manual_panel = widgets.VBox([edge_help, edge_entry])
csv_panel = widgets.VBox([edge_help, upload])

seed_entry = widgets.IntText(value=7, description='Seed:', style={'description_width':'initial'})
time_entry = widgets.BoundedFloatText(value=12.0, min=0.1, max=200,
                                      description='Final time:', style={'description_width':'initial'})
spread_entry = widgets.BoundedFloatText(value=0.05, min=0, max=5,
                                        description='Initial spread:', style={'description_width':'initial'})
run_button = widgets.Button(description='Validate & simulate', button_style='success', icon='play')
save_button = widgets.Button(description='Export CSV + PNG', icon='download', disabled=True)
report = widgets.Output()
LAST_RUN = None

def refresh_panels(*_):
    custom_panel.layout.display = '' if oscillator.value == 'User-defined' else 'none'
    manual_panel.layout.display = '' if graph_mode.value == 'Paste directed edges' else 'none'
    csv_panel.layout.display = '' if graph_mode.value == 'Upload edge-list CSV' else 'none'

oscillator.observe(refresh_panels, names='value')
graph_mode.observe(refresh_panels, names='value')
refresh_panels()

def run_clicked(_):
    global LAST_RUN
    save_button.disabled = True
    LAST_RUN = None
    with report:
        clear_output(wait=True)
        try:
            n, seed = n_vertices.value, seed_entry.value
            cfg = oscillator_choice(oscillator.value, p_entry.value, a_entry.value)
            validate_oscillator(cfg)
            if graph_mode.value == 'Random eligible graph':
                G = random_eligible_graph(n, seed)
            elif graph_mode.value == 'Paste directed edges':
                G = parse_edge_text(edge_entry.value, n)
            else:
                G = graph_from_uploaded_csv(upload, n)

            order = source_scc_order(G)  # Reject invalid graphs and STOP here.
            print(f'✓ Graph accepted: {len(G)} vertices, {G.number_of_edges()} directed edges; {len(order)} SCCs.')
            print(f'✓ Dynamics: {cfg.name}; a = {cfg.a:.6g}; P diagonal = {cfg.p.tolist()}')
            print(f'  {cfg.note}')
            G, checks = allocate_weights(G, cfg.a, seed)
            result = verify_weights(G, checks, cfg.a)
            if not result['positive'] or not result['imbalance']:
                raise RuntimeError('Weight verification failed; simulation stopped.')
            print(f"✓ All weights positive; non-root imbalance ≤ -2a.")
            print(f"  Weight range: [{result['min_weight']:.4g}, {result['max_weight']:.4g}]")
            if result['max_weight'] > 1e6:
                print('⚠ Large weights may make numerical integration very stiff.')
            t, X, A = simulate(G, cfg, time_entry.value, seed, spread_entry.value)
            root = checks[0]['root']
            edges, D = tree_distances(G, root, X)
            print(f'✓ Final mean tree-edge error: {D[-1].mean():.3g}; max: {D[-1].max():.3g}')
            fig = make_plot(G, t, D, seed)
            display(fig)
            plt.close(fig)
            LAST_RUN = dict(G=G, t=t, D=D, A=A, edges=edges, fig=fig)
            save_button.disabled = False
        except (ValueError, nx.NetworkXException, RuntimeError, TypeError, OverflowError) as exc:
            print(f'⚠ STOPPED: {exc}')
        except Exception as exc:
            print(f'⚠ STOPPED: {type(exc).__name__}: {exc}')
            print('Check the custom dynamics cell and your graph input.')

def save_clicked(_):
    with report:
        if LAST_RUN is not None:
            print('Saved to:', export_results(LAST_RUN).resolve())

run_button.on_click(run_clicked)
save_button.on_click(save_clicked)

display(widgets.VBox([
    widgets.HTML('<h3 style="margin:0 0 5px">Graph-Theoretic Synchronization</h3>'),
    oscillator, custom_panel,
    widgets.HTML('<b>Directed graph</b>'),
    widgets.HBox([graph_mode, n_vertices]), manual_panel, csv_panel,
    widgets.HTML('<b>Simulation</b>'),
    widgets.HBox([seed_entry, time_entry, spread_entry]),
    widgets.HBox([run_button, save_button]), report
]))

### What the checks do—and do not do

- **Checked:** one source SCC (equivalent to an out-arborescence); positive assigned weights; required internal imbalance inequalities.
- **Assumed:** a valid global QUAD inequality for the selected dynamics. User-defined entries of `P` and `a` are **not** enough to establish this.
- **Illustrated numerically:** decay of distances along a spanning out-arborescence. A finite-time plot is not a proof of asymptotic synchronization.

**Inputs:** directed edge list `tail,head`; vertices `0,…,N−1`. **Output:** weighted adjacency, directed edge weights, tree-edge errors, PNG graph/error figure. See `README.md` for details.